In [4]:
"""
Extract CHM (canopy height model) summary stats and 0.5 m histograms
for BLM AIM Riparian Wetland polygons in Nevada.
 
CHM source: projects/naip-chm/assets/conus-structure-model
  - Native resolution: 0.6 m
  - Band: B0 (height x 100, so divide by 100 to get meters)
  - Tiling: NAIP quarter-quads, in local UTM zone per tile
 
AIM polygons: projects/dri-apps/assets/blm-riparian/aim-rw-footprints-V3-NV_20250425
  - ~350 small polygons across NV
  - NV is in UTM Zone 11N (EPSG:26911)
 
Fix vs prior version: filter the IC to NV BEFORE pulling a projection or
mosaicking, and explicitly set the projection to Zone 11 to prevent
on-the-fly reprojection into Zone 17 (which was inherited from the
alphabetically-first image in the unfiltered IC).
 
Output: CSV with per-polygon mean, median, max, count, and 60 histogram
bins (0-30 m, 0.5 m wide).
"""

import ee

ee.Initialize(project='dri-blm')

# ---------------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------------
RW_ASSET = 'projects/dri-apps/assets/blm-riparian/aim-rw-footprints-V3-NV_20250425'
CHM_IC = 'projects/naip-chm/assets/conus-structure-model'

# Histogram parameters (0.5 m bins from 0 to 30 m -> 60 bins)
HIST_MIN = 0.0
HIST_MAX = 30.0
BIN_WIDTH = 0.5
N_BINS = int((HIST_MAX - HIST_MIN) / BIN_WIDTH)  # 60

# Native CHM resolution; honor this for the reduction
SCALE = 0.6

# tileScale: bump up for many small polygons to reduce per-tile memory pressure
TILE_SCALE = 4

# Identifier field on the AIM polygons. Adjust if the FC uses a different
# primary key (common AIM names: 'PlotID', 'PrimaryKey', 'PlotKey').
PLOT_ID_FIELD = 'Evaluation'

# Output
OUTPUT_DESCRIPTION = 'aim_rw_chm_stats_V3_NV'
OUTPUT_FOLDER = 'MESIC_CHM'

# ---------------------------------------------------------------------------
# Load FC and filter the IC spatially first
# ---------------------------------------------------------------------------
rw = ee.FeatureCollection(RW_ASSET)
rw_bounds = rw.geometry().bounds()
 
chm_ic_nv = (
    ee.ImageCollection(CHM_IC)
      .filterBounds(rw_bounds)
      .select(['B0'], ['chm_cm'])
)
 
nv_projection = ee.Image(chm_ic_nv.first()).projection()
 
# ---------------------------------------------------------------------------
# Build the CHM image with explicit projection
# ---------------------------------------------------------------------------
chm = (
    chm_ic_nv.mosaic()
             .divide(100)
             .rename('chm_m')
             .toFloat()
             .setDefaultProjection(nv_projection)
)
 
# ---------------------------------------------------------------------------
# Combined reducer -- all unweighted for consistency
# ---------------------------------------------------------------------------
reducer = (
    ee.Reducer.mean().unweighted()
      .combine(ee.Reducer.median().unweighted(), sharedInputs=True)
      .combine(ee.Reducer.max().unweighted(), sharedInputs=True)
      .combine(ee.Reducer.count().unweighted(), sharedInputs=True)
      .combine(
          ee.Reducer.fixedHistogram(HIST_MIN, HIST_MAX, N_BINS).unweighted(),
          sharedInputs=True,
      )
)
 
# ---------------------------------------------------------------------------
# Reduce
# ---------------------------------------------------------------------------
stats_fc = chm.reduceRegions(
    collection=rw,
    reducer=reducer,
    scale=SCALE,
    tileScale=TILE_SCALE,
    crs='EPSG:26911',
)
 
# ---------------------------------------------------------------------------
# Flatten histogram into named bin columns
# ---------------------------------------------------------------------------
bin_edges = [HIST_MIN + i * BIN_WIDTH for i in range(N_BINS)]
bin_names = [
    'bin_{:.1f}_{:.1f}'.format(e, e + BIN_WIDTH).replace('.', 'p')
    for e in bin_edges
]
 
 
def flatten_hist(feat):
    feat = ee.Feature(feat)
    hist = feat.get('histogram')
    counts = ee.Algorithms.If(
        hist,
        ee.Array(hist).slice(1, 1, 2).project([0]).toList(),
        ee.List.repeat(0, N_BINS),
    )
    bin_dict = ee.Dictionary.fromLists(bin_names, ee.List(counts))
    return feat.set(bin_dict).set('histogram', None)
 
 
stats_flat = stats_fc.map(flatten_hist)
 
# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------
selectors = [PLOT_ID_FIELD, 'mean', 'median', 'max', 'count'] + bin_names
 
task = ee.batch.Export.table.toDrive(
    collection=stats_flat,
    description=OUTPUT_DESCRIPTION,
    folder=OUTPUT_FOLDER,
    fileNamePrefix=OUTPUT_DESCRIPTION,
    fileFormat='CSV',
    selectors=selectors,
)
 
task.start()
print('Export task started:', task.id)

Export task started: Q2WQT27W34FA7VT72HWSEI5Q
